In [3]:
print(123)

123


In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from openai import OpenAI
openai_client = OpenAI()

# Plain LLMs lack our data

In [6]:
# First, let's define a function to talk to the LLM. This LLM fucntion lets us to interact with the LLM provider

In [7]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [8]:
llm("Hey, what's up?")

'Not much—just here and ready to help. What’s on your mind?'

In [9]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes, you may still be able to join, depending on the course’s enrollment rules and whether it’s still open.

A few things to check:
- **Registration deadline**: Some courses allow late enrollment, some don’t.
- **Seat availability**: The class may be full.
- **Prerequisites**: You may need prior approval if required.
- **Instructor/department approval**: Some courses need permission to add after the start date.

If you want, I can help you draft a short message to the instructor or registrar asking whether you can still enroll.


In [10]:
#These are the things we think will be helpful for the LLM to answer the question

context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [11]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [12]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [13]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [14]:
# RAG Architecture 

# That's the entire architecture. It comes down to three components.

# The pieces are search, the prompt, and the LLM:

# search
# prompt
# LLM

In [15]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [16]:
# The FAQ data is available as JSON from the DataTalks.Club website. The FAQ data has been made available as a JSON endpoint 
# Now lets fetch the data

In [17]:


import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [19]:
courses_raw # This returns a list of courses. Each course has a path field that points to its FAQ data.

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 103},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [20]:
# This code loops through a list of courses and downloads/collects FAQ data for each one from a set of URLs. 
documents = [] # Creates an empty list that will hold all the collected FAQ documents from every course.
url_prefix = "https://datatalks.club/faq" # The base URL that all course-specific URLs will be built from.

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status() #raise an issue if something isn't working. Dont continue.
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1368

In [21]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [ ]:
# We dot not want to send all documents to LLM cos would be very expensive and not effective 
# We will index the data so only the most relevant docs are retrieved 
# Searching matters because where you have loads of documents. Sending all of them to the LLM would be expensive and slow. 
# The model would get confused with too much data. Search finds the most relevant documents to send instead.

In [22]:
# There are many search libraries you can use - Apache Lucene, Elasticsearch, Solr, and others. 
# But these are somewhat heavy. For example, to run Elasticsearch, you need to start a Docker container.
# Elastic search is based on Lucene. So it uses Lucene under the hood
# There is a light weight alternatice called minsearch. minsearch is a simple in-memory search engine. It's lightweight, 
# so it runs anywhere Python runs, including Google Colab where you can't start a Docker container. 
# It's a toy implementation, not production ready, but it illustrates how search engines work and it gives good results.

In [24]:
# Text fields are the fields you search through. When you type a query, the search engine looks at these fields and tokenizes them. 
# It splits text into words, lowercases them, removes stop words, and ranks the results by relevance. So question, section, and answer 
# are text fields.

# Keyword fields are for exact matching. Think of a SQL query like SELECT * FROM index WHERE course = 'data-engineering-zoomcamp'. 
# The results must come from the specified course, no matter what ranking or boosting you do for text fields.


from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [27]:
# index.search(question)

In [28]:
index.search (question, filter_dict={'course': 'llm-zoomcamp'},
              num_results = 5)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [25]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [ ]:
# We used boost_dict to give the question field more weight (2.0 instead of the default 1.0) and section less weight (0.5). 
# Query words appearing in the question field is a stronger signal than them appearing in the section name.
# We used filter_dict to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )